# Bank Statement CSV Processor

A program to load a CSV or PDF bank statement, if PDF it converts to CSV then display as a table, and filter by purchase category.

In [1]:
# Step 1: Define a reusable function to process bank statements
import pandas as pd
import os
import re

# This function will be used in Step 3 to filter transactions by category.
# Input and define your own categories and keywords as needed.
def infer_category(description):
    """Infer a category from a transaction description when none is provided."""
    text = str(description).lower()
    category_keywords = {
        'groceries': ['grocery', 'supermarket', 'market', 'aldi', 'walmart', 'target', 'winn-dixi', 'publix'],
        'dining': ['taco', 'starbucks', 'restaurant', 'cafe', 'coffee', 'pizza', 'diner', 'doordash', 'uber eats', "mcdonald's"],
        'entertainment': ['netflix', 'audible', 'spotify', 'hulu', 'cinema', 'movie', 'game'],
        'hobbies/crafts': ['staples', 'goodwill', 'fabric', 'craft', 'yarn', 'hobbylobb', 'sewing', 'studio', 'etsy', 'dollartre', 'dollar'],
        'transport': ['uber', 'lyft', 'gas', 'fuel', 'shell', 'exxon', 'wawa', '7-eleven', 'racetrac', 'e-pass', 'toll', 'parking'],
        'utilities': ['electric', 'water', 'internet', 'phone', 'utility'],
        'shopping': ['amazon', 'store', 'shop', 'purchase', 'afterpay'],
        'health': ['pharmacy', 'doctor', 'medical', 'clinic', 'fitness', 'gym'],
        'deposit': ['deposit', 'payroll', 'direct deposit', 'refund']
    }

    for category, keywords in category_keywords.items():
        if any(keyword in text for keyword in keywords):
            return category
    return 'other'


def _looks_like_header(value):
    """Return True if a value looks like a real column header (not transaction data)."""
    s = str(value).strip()
    # A real header typically won't start with a date pattern or contain a dollar amount
    has_date = bool(re.match(r'^\d{2}/\d{2}', s))
    has_amount = bool(re.search(r'[+-]?\d+\.\d{2}\s*$', s))
    return not has_date and not has_amount


def process_bank_statement(csv_path, categories_to_exclude=None, save_output=True):
    """Process a bank statement CSV and filter by categories."""

    if categories_to_exclude is None:
        categories_to_exclude = ['none', '']

    if not os.path.exists(csv_path):
        print(f"✗ CSV file not found: {csv_path}")
        return None

    try:
        # Peek at the first row to decide whether it's a real header
        peek = pd.read_csv(csv_path, nrows=0)
        first_col = peek.columns[0] if len(peek.columns) > 0 else ''

        if _looks_like_header(first_col):
            bank_df = pd.read_csv(csv_path)
        else:
            # No header row — first row is real data; assign column names
            bank_df = pd.read_csv(csv_path, header=None)
            bank_df.columns = [f'col_{i}' for i in range(len(bank_df.columns))]
            print("! Detected headerless CSV. Assigned generic column names.\n")

        print(f"✓ CSV loaded: {csv_path}")
        print(f"  Shape: {bank_df.shape}")
        print(f"  Columns: {list(bank_df.columns)}\n")
    except Exception as e:
        print(f"✗ Error loading CSV: {e}")
        return None

    # Remove rows containing known bank portal text noise.
    bank_name = 'FAIRWINDS'
    bank_noise_patterns = ['www.fairwinds.org', 'fairwinds']
    bank_noise_mask = bank_df.astype(str).apply(
        lambda col: col.str.contains('|'.join(bank_noise_patterns), case=False, na=False)
    ).any(axis=1)
    removed_bank_noise = int(bank_noise_mask.sum())
    if removed_bank_noise:
        bank_df = bank_df[~bank_noise_mask].copy()
        print(f"! Removed {removed_bank_noise} rows containing {bank_name} text noise.\n")

    category_column = None
    for col in bank_df.columns:
        if 'category' in str(col).lower():
            category_column = col
            break

    # Fallback for one-column transaction exports with no explicit category.
    if category_column is None and len(bank_df.columns) == 1:
        only_col = bank_df.columns[0]
        bank_df['description'] = bank_df[only_col].astype(str)
        bank_df['amount'] = bank_df['description'].str.extract(r'([+-]?\d+\.\d{2})\s*$')[0]
        bank_df['inferred_category'] = bank_df['description'].apply(infer_category)
        category_column = 'inferred_category'
        print("! No category column found. Generated 'inferred_category' from description text.\n")

    if not category_column:
        print("✗ Could not find or infer a category column")
        print(f"Available columns: {list(bank_df.columns)}")
        return None

    excluded_categories = [cat.lower() for cat in categories_to_exclude]
    filtered_df = bank_df[
        ~bank_df[category_column].astype(str).str.lower().isin(excluded_categories)
    ]

    print("✓ Filtering complete:")
    print(f"  Category column: '{category_column}'")
    print(f"  Original rows: {len(bank_df)}")
    print(f"  Filtered rows: {len(filtered_df)}")
    print(f"  Rows removed: {len(bank_df) - len(filtered_df)}\n")

    if save_output:
        output_path = os.path.join(os.path.dirname(csv_path), 'filtered_' + os.path.basename(csv_path))
        filtered_df.to_csv(output_path, index=False)
        print(f"✓ Saved to: {output_path}\n")

    return filtered_df

In [2]:
# Step 2: Accept either a PDF or CSV input file
import os

input_path = r'C:\Users\chels\OneDrive\Desktop\Software Dev Projects\Software-Dev-Projects\Automated Expenses Tracker\june2025output.csv'
csv_from_pdf = r'C:\Users\chels\Downloads\bankstatement_from_pdf.csv'

if not os.path.exists(input_path):
    raise FileNotFoundError(f'Input file not found: {input_path}')

file_ext = os.path.splitext(input_path)[1].lower()

if file_ext == '.pdf':
    try:
        import pdfplumber
    except ImportError:
        import sys
        !{sys.executable} -m pip install pdfplumber --prefer-binary
        import pdfplumber

    try:
        rows = []
        with pdfplumber.open(input_path) as pdf:
            for page in pdf.pages:
                table = page.extract_table()
                if table:
                    rows.extend(table)

        if not rows:
            raise ValueError('No table data was found in the PDF.')

        df_pdf = pd.DataFrame(rows[1:], columns=rows[0])
        df_pdf.to_csv(csv_from_pdf, index=False)

        csv_path = csv_from_pdf
        print(f'✓ PDF detected. Converted to CSV: {csv_path}')
        print(f'  Rows saved: {len(df_pdf)}')
    except Exception as e:
        print('✗ PDF conversion failed:', e)
        raise

elif file_ext == '.csv':
    csv_path = input_path
    print(f'✓ CSV detected. Using file directly: {csv_path}')

else:
    raise ValueError(f'Unsupported file type: {file_ext}. Please use a .pdf or .csv file.')

✓ CSV detected. Using file directly: C:\Users\chels\OneDrive\Desktop\Software Dev Projects\Software-Dev-Projects\Automated Expenses Tracker\june2025output.csv


In [3]:
# Step 3: Process whichever CSV path was resolved in Step 2
categories = ['groceries', 'utilities', 'health', 'dining', 'entertainment']  # Example categories to exclude

# Call the function
filtered_df = process_bank_statement(csv_path, categories_to_exclude=categories)

# For a different file, change input_path in Step 2 and re-run Step 2 then Step 3.

! Detected headerless CSV. Assigned generic column names.

✓ CSV loaded: C:\Users\chels\OneDrive\Desktop\Software Dev Projects\Software-Dev-Projects\Automated Expenses Tracker\june2025output.csv
  Shape: (238, 1)
  Columns: ['col_0']

! Removed 5 rows containing FAIRWINDS text noise.

! No category column found. Generated 'inferred_category' from description text.

✓ Filtering complete:
  Category column: 'inferred_category'
  Original rows: 233
  Filtered rows: 197
  Rows removed: 36

✓ Saved to: C:\Users\chels\OneDrive\Desktop\Software Dev Projects\Software-Dev-Projects\Automated Expenses Tracker\filtered_june2025output.csv



In [4]:
# Step 4: View filtered data organized by category with totals
if filtered_df is None:
    print('No filtered data available. Step 3 likely failed to find a category column.')
else:
    # Determine which column holds the description and amount
    desc_col = 'description' if 'description' in filtered_df.columns else filtered_df.columns[0]
    amount_col = 'amount' if 'amount' in filtered_df.columns else None
    cat_col = 'inferred_category' if 'inferred_category' in filtered_df.columns else None
    for col in filtered_df.columns:
        if 'category' in str(col).lower():
            cat_col = col
            break

    if cat_col is None:
        print('No category column found in filtered data.')
    else:
        grand_total = 0.0
        for category, group in filtered_df.groupby(cat_col):
            print(f"\n{'='*60}")
            print(f"  Category: {category.upper()}  ({len(group)} transactions)")
            print(f"{'='*60}")
            print(f"  {'#':<5} {'Description':<45} {'Amount':>10}")
            print(f"  {'-'*5} {'-'*45} {'-'*10}")
            cat_total = 0.0
            for i, (_, row) in enumerate(group.iterrows(), 1):
                desc = str(row[desc_col])[:44]
                amt_raw = row[amount_col] if amount_col else None
                try:
                    amt = float(amt_raw)
                    amt_str = f"{amt:>10.2f}"
                    cat_total += amt
                except (TypeError, ValueError):
                    amt_str = f"{'N/A':>10}"
                print(f"  {i:<5} {desc:<45} {amt_str}")
            print(f"  {'':5} {'CATEGORY TOTAL':<45} {cat_total:>10.2f}")
            grand_total += cat_total

        print(f"\n{'='*60}")
        print(f"  {'GRAND TOTAL':<50} {grand_total:>10.2f}")
        print(f"{'='*60}")


  Category: DEPOSIT  (9 transactions)
  #     Description                                       Amount
  ----- --------------------------------------------- ----------
  1     06/10 ACH Deposit LIBERTY MUTUAL - TDU ACH P      301.52
  2     06/17 Deposit CASH APP*RUANA RAE*CASHOakland      140.00
  3     06/21 Deposit CASH APP*RUANA RAE*CASHOakland       14.74
  4     06/21 Deposit VENMO*Rae Ruana New York CityN      317.35
  5     06/24 Deposit Ebay Commerce Inc. San Jose CA       19.17
  6     06/25 ACH Deposit eventbrite.com - 128640813       14.55
  7     06/27 ACH Deposit LIBERTY MUTUAL - TDU ACH P      301.52
  8     06/28 Deposit aliexpress 408-7855580 CAUS 3.        3.46
  9     169 Withdrawals = 4,972.90 11 Deposits = 4,1      163.14
        CATEGORY TOTAL                                   1275.45

  Category: HOBBIES/CRAFTS  (24 transactions)
  #     Description                                       Amount
  ----- --------------------------------------------- ----------
  1 

In [5]:
# Step 5: Category totals summary
if filtered_df is None:
    print('No filtered data to display. Please check Step 3 output.')
else:
    cat_col = 'inferred_category' if 'inferred_category' in filtered_df.columns else None
    for col in filtered_df.columns:
        if 'category' in str(col).lower():
            cat_col = col
            break
    amount_col = 'amount' if 'amount' in filtered_df.columns else None

    if cat_col and amount_col:
        summary = (
            filtered_df.groupby(cat_col)[amount_col]
            .apply(lambda x: pd.to_numeric(x, errors='coerce').sum())
            .reset_index()
        )
        summary.columns = ['Category', 'Total Amount']
        summary = summary.sort_values('Total Amount')
        summary['Total Amount'] = summary['Total Amount'].map(lambda x: f"${x:,.2f}")
        print("Category Totals Summary:")
        print(summary.to_string(index=False))
    elif cat_col:
        counts = filtered_df[cat_col].value_counts().reset_index()
        counts.columns = ['Category', 'Transaction Count']
        print("Category Counts (no amount column available):")
        print(counts.to_string(index=False))

Category Totals Summary:
      Category Total Amount
         other   $-3,083.51
      shopping     $-405.61
hobbies/crafts     $-318.82
     transport     $-289.11
       deposit    $1,275.45


In [6]:
# Inspect 'other' category transactions
cat_col = 'inferred_category' if 'inferred_category' in filtered_df.columns else None
desc_col = 'description' if 'description' in filtered_df.columns else filtered_df.columns[0]
amount_col = 'amount' if 'amount' in filtered_df.columns else None

other_df = filtered_df[filtered_df[cat_col] == 'other']
print(f"'Other' category: {len(other_df)} transactions\n")
for i, (_, row) in enumerate(other_df.iterrows(), 1):
    amt = f"  {float(row[amount_col]):>10.2f}" if amount_col and row[amount_col] == row[amount_col] else "         N/A"
    print(f"{i:<4} {str(row[desc_col]):<55}{amt}")

'Other' category: 118 transactions

1    06/01 Withdrawal Debit EVENTBRITE AD CAMPAIGN EVENTBRITE.COCAUS -15.32      -15.32
2    06/01 Withdrawal Debit FACEBK *CE57BS4PR2 6505434800 CAUS -5.00       -5.00
3    MCCULLOCOVIEDO F                                                N/A
4    06/02 Withdrawal Debit TST*SANTIAGOS ALTAMONTEAltamonte SprFLUS -61.31      -61.31
5    STEORLANDO F                                                    N/A
6    06/03 Withdrawal Debit BP#9492828CIRCL ORLANDO ORLANDO -10.05      -10.05
7    FLUS                                                            N/A
8    ORLANDO F                                                       N/A
9    ORLANDO F                                                       N/A
10   ORLANDO F                                                       N/A
11   06/05 Withdrawal Debit BP#9492828CIRCL ORLANDO ORLANDO -3.08       -3.08
12   FLUS                                                            N/A
13   STEORLANDO F                      